# 07. Bronze annotation export

Отдельный ноутбук для подготовки большой `bronze`-разметки из годов Lenta, которые не участвовали в `golden` / `silver` / текущем candidate pool.

Цель ноутбука:

```text
lenta_clean_news.csv за другие годы
→ исключение golden/silver/current candidate ids
→ BGE-M3 embeddings
→ legacy threshold graph clustering
→ bronze candidates с previous_items context
→ CSV batches + instructions + manifest + zip
```

Важное ограничение: этот ноутбук **не обучает модель** и **не меняет таблицу экспериментов**. Он только готовит данные для дальнейшей LLM/ручной bronze-разметки.

Первый запуск может быть долгим: считаются embeddings для новых годов. Если тяжело, уменьшить `LARGE_BRONZE_YEARS` или `MAX_NEWS_PER_YEAR` в конфигурации.

## 1. Imports, paths, encoder

Если ноутбук запускается не из папки `notebooks`, поправить `PROJECT_ROOT`.

In [ ]:
from __future__ import annotations

import json
import re
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd


# Обычно ноутбук лежит в PROJECT_ROOT/notebooks, поэтому Path("..").
PROJECT_ROOT = Path("..").resolve()

# Fallback: если ноутбук открыт прямо из корня проекта.
if not (PROJECT_ROOT / "src").exists() and (Path.cwd() / "src").exists():
    PROJECT_ROOT = Path.cwd().resolve()

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

%load_ext autoreload
%autoreload 2

from model.config import ExperimentPaths, SemanticNewsBaselineConfig
from model.embeddings import SentenceTransformerEncoder

paths = ExperimentPaths(project_root=PROJECT_ROOT)

paths.artifacts_dir.mkdir(parents=True, exist_ok=True)
paths.prepared_dir.mkdir(parents=True, exist_ok=True)

baseline_config = SemanticNewsBaselineConfig()

encoder = SentenceTransformerEncoder(
    model_name=baseline_config.embedding_model_name,
    batch_size=16,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Prepared dir:", paths.prepared_dir)
print("Artifacts dir:", paths.artifacts_dir)
print("Embedding model:", baseline_config.embedding_model_name)

## 2. Конфигурация bronze export

Для первого тестового запуска можно поставить:

```python
LARGE_BRONZE_YEARS = [2005]
MAX_NEWS_PER_YEAR = 5000
LARGE_BRONZE_MAX_CANDIDATES = 1000
```

Для нормального прогона можно расширять годы и лимиты.

In [ ]:
# ---------------------------------------------------------------------
# Large bronze export config
# ---------------------------------------------------------------------

LARGE_BRONZE_INPUT_PATH = paths.prepared_dir / "lenta_clean_news.csv"

# Годы, которые не входят в текущий golden/silver candidate pool.
LARGE_BRONZE_YEARS = [2005, 2006]

# Защита от слишком тяжёлого первого запуска.
# Если всё работает стабильно — можно поднять лимит или поставить None.
MAX_NEWS_PER_YEAR = 12000

# Сколько строк-кандидатов хотим получить на выходе.
LARGE_BRONZE_MAX_CANDIDATES = 5000

LARGE_BRONZE_BATCH_SIZE = 100
LARGE_BRONZE_MAX_PREVIOUS_ITEMS = 4
LARGE_BRONZE_MAX_TEXT_CHARS = 900
LARGE_BRONZE_RANDOM_STATE = 42

# Кластеризация как в зафиксированном baseline.
STORY_THRESHOLD = 0.82
STORY_WINDOW_DAYS = 14
USE_TOPIC_BLOCKING = True

# Current rows всегда исключают golden/silver/current candidate pool.
# Context rows можно не исключать: они только контекст, а не train labels.
# Для полной паранойи можно поставить True, но кандидатов будет меньше.
LARGE_BRONZE_STRICT_CONTEXT_EXCLUDES = False

LARGE_BRONZE_OUTPUT_DIR = paths.artifacts_dir / "bronze_annotation_large"
LARGE_BRONZE_BATCH_DIR = LARGE_BRONZE_OUTPUT_DIR / "batches"
LARGE_BRONZE_PACKAGE_PATH = LARGE_BRONZE_OUTPUT_DIR / "bronze_annotation_large_package.zip"

print("Large bronze input:", LARGE_BRONZE_INPUT_PATH)
print("Years:", LARGE_BRONZE_YEARS)
print("Max news per year:", MAX_NEWS_PER_YEAR)
print("Max candidates:", LARGE_BRONZE_MAX_CANDIDATES)
print("Output dir:", LARGE_BRONZE_OUTPUT_DIR)

## 3. Helper-функции

Ячейка самодостаточная: нормализация `news_id`, сбор no-leakage ids, подготовка clean-like входа, простая legacy graph clustering, сбор `previous_items`, экспорт батчей.

In [ ]:
# ---------------------------------------------------------------------
# Utility functions
# ---------------------------------------------------------------------

def normalize_news_id_for_join(series: pd.Series) -> pd.Series:
    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def truncate_text(value, max_chars: int = 900) -> str:
    text = "" if pd.isna(value) else str(value)
    text = " ".join(text.split())

    if len(text) <= max_chars:
        return text

    return text[:max_chars].rstrip() + "..."


def get_text_value(row: pd.Series) -> str:
    if "text" in row.index:
        return row.get("text", "")

    if "model_text" in row.index:
        return row.get("model_text", "")

    return ""


def l2_normalize_np(matrix: np.ndarray) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return matrix / norms


def count_words(value: str) -> int:
    value = "" if pd.isna(value) else str(value)
    return len(re.findall(r"\w+", value, flags=re.UNICODE))


def prepare_clean_like_for_bronze(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Prepare clean-like input for bronze clustering/embedding."""
    df = raw_df.copy()

    required_columns = ["news_id", "published_at", "topic", "title", "text"]
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    df["news_id"] = normalize_news_id_for_join(df["news_id"])
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    df["topic"] = df["topic"].fillna("unknown").astype(str)
    df["title"] = df["title"].fillna("").astype(str)
    df["text"] = df["text"].fillna("").astype(str)

    if df["published_at"].isna().any():
        bad_count = int(df["published_at"].isna().sum())
        raise ValueError(f"Found invalid published_at rows: {bad_count}")

    if "model_text" not in df.columns:
        df["model_text"] = df["title"] + "\n\n" + df["text"]
    else:
        df["model_text"] = df["model_text"].fillna("").astype(str)

    # Recompute technical columns if absent. They are useful for downstream consistency.
    if "title_length" not in df.columns:
        df["title_length"] = df["title"].str.len()
    if "title_num_words" not in df.columns:
        df["title_num_words"] = df["title"].map(count_words)
    if "text_length" not in df.columns:
        df["text_length"] = df["text"].str.len()
    if "text_num_words" not in df.columns:
        df["text_num_words"] = df["text"].map(count_words)
    if "model_length" not in df.columns:
        df["model_length"] = df["model_text"].str.len()
    if "model_num_words" not in df.columns:
        df["model_num_words"] = df["model_text"].map(count_words)
    if "url" not in df.columns:
        df["url"] = ""
    if "tags" not in df.columns:
        df["tags"] = ""

    df["_news_id_sort_key"] = pd.to_numeric(df["news_id"], errors="coerce")

    df = (
        df
        .sort_values(["published_at", "_news_id_sort_key", "news_id"], kind="mergesort")
        .drop(columns=["_news_id_sort_key"])
        .reset_index(drop=True)
    )

    return df


def collect_ids_from_frame(
    frame: pd.DataFrame | None,
    frame_name: str,
    id_column: str = "news_id",
) -> set[str]:
    if frame is None:
        return set()

    if id_column not in frame.columns:
        print(f"[WARN] {frame_name}: no {id_column} column, skipped")
        return set()

    ids = set(normalize_news_id_for_join(frame[id_column]))
    print(f"{frame_name}: collected {len(ids)} ids")
    return ids


def read_ids_from_csv(path: Path, frame_name: str, id_column: str = "news_id") -> set[str]:
    if not path.exists():
        return set()

    try:
        frame = pd.read_csv(path, usecols=[id_column])
        return collect_ids_from_frame(frame, frame_name, id_column=id_column)
    except Exception as error:
        print(f"[WARN] failed to read ids from {path}: {error}")
        return set()


def collect_silver_ids(paths: ExperimentPaths) -> set[str]:
    """Collect silver ids from known annotation/features files."""
    silver_ids = set()

    known_silver_paths = [
        paths.silver_path,
        paths.prepared_dir / "lenta_silver_set_llm.csv",
        paths.prepared_dir / "lenta_silver_set.csv",
        paths.prepared_dir / "lenta_silver_annotations.csv",
        paths.prepared_dir / "lenta_llm_silver_set.csv",
        paths.artifacts_dir / "significance_model" / "silver_significance_features.csv",
        getattr(paths, "significance_model_dir", paths.artifacts_dir / "significance_model") / "silver_significance_features.csv",
    ]

    seen_paths = set()
    for silver_path in known_silver_paths:
        silver_path = Path(silver_path)
        if silver_path in seen_paths:
            continue
        seen_paths.add(silver_path)
        silver_ids |= read_ids_from_csv(silver_path, silver_path.name)

    print(f"Total silver ids: {len(silver_ids)}")
    return silver_ids


class UnionFind:
    def __init__(self, n: int) -> None:
        self.parent = np.arange(n)
        self.rank = np.zeros(n, dtype=np.int8)

    def find(self, x: int) -> int:
        parent = self.parent
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return int(x)

    def union(self, a: int, b: int) -> bool:
        root_a = self.find(a)
        root_b = self.find(b)

        if root_a == root_b:
            return False

        if self.rank[root_a] < self.rank[root_b]:
            root_a, root_b = root_b, root_a

        self.parent[root_b] = root_a

        if self.rank[root_a] == self.rank[root_b]:
            self.rank[root_a] += 1

        return True


def legacy_threshold_graph_cluster_ids(
    news_df: pd.DataFrame,
    embeddings: np.ndarray,
    similarity_threshold: float = 0.82,
    window_days: int = 14,
    use_topic_blocking: bool = True,
) -> tuple[pd.Series, int]:
    """Legacy-like threshold graph clustering.

    Edges are added when two news items are close enough semantically and temporally.
    Connected components become cluster ids.
    """
    news = news_df.copy().reset_index(drop=True)
    news["published_at"] = pd.to_datetime(news["published_at"], errors="coerce")

    if len(news) != len(embeddings):
        raise ValueError(f"News/embeddings mismatch: {len(news)} != {len(embeddings)}")

    normalized_embeddings = l2_normalize_np(embeddings)

    news["_row_pos"] = np.arange(len(news))

    uf = UnionFind(len(news))
    edge_count = 0
    window_delta = np.timedelta64(window_days, "D")

    if use_topic_blocking and "topic" in news.columns:
        group_iterator = news.groupby("topic", sort=False)
    else:
        group_iterator = [("all", news)]

    for _, group_df in group_iterator:
        group_df = group_df.sort_values(["published_at", "_row_pos"], kind="mergesort")

        indices = group_df["_row_pos"].astype(int).to_numpy()
        dates = group_df["published_at"].to_numpy(dtype="datetime64[ns]")

        start = 0

        for local_i in range(len(indices)):
            idx_i = int(indices[local_i])
            date_i = dates[local_i]

            while start < local_i and date_i - dates[start] > window_delta:
                start += 1

            if start == local_i:
                continue

            candidate_indices = indices[start:local_i].astype(int)

            if len(candidate_indices) == 0:
                continue

            similarities = normalized_embeddings[candidate_indices] @ normalized_embeddings[idx_i]
            connected_indices = candidate_indices[similarities >= similarity_threshold]

            for idx_j in connected_indices:
                if uf.union(idx_i, int(idx_j)):
                    edge_count += 1

    root_to_cluster_id = {}
    cluster_ids = []

    for row_pos in range(len(news)):
        root = uf.find(row_pos)

        if root not in root_to_cluster_id:
            root_to_cluster_id[root] = f"cluster_{len(root_to_cluster_id):06d}"

        cluster_ids.append(root_to_cluster_id[root])

    return pd.Series(cluster_ids, index=news_df.index, name="cluster_id"), edge_count


def build_bronze_annotation_candidates(
    clustered_news: pd.DataFrame,
    embeddings: np.ndarray,
    exclude_current_ids: set[str],
    exclude_context_ids: set[str] | None = None,
    max_rows: int = 5000,
    max_previous_items: int = 4,
    max_text_chars: int = 900,
    random_state: int = 42,
) -> pd.DataFrame:
    """Build candidate rows for bronze annotation.

    Each current article gets several previous articles from the same cluster as context.
    """
    news = clustered_news.copy().reset_index(drop=True)

    required_columns = {"news_id", "published_at", "cluster_id", "title"}
    missing_columns = required_columns - set(news.columns)

    if missing_columns:
        raise ValueError(f"clustered_news is missing required columns: {sorted(missing_columns)}")

    if len(news) != len(embeddings):
        raise ValueError(f"News/embeddings length mismatch: {len(news)} != {len(embeddings)}")

    news["news_id"] = normalize_news_id_for_join(news["news_id"])
    news["cluster_id"] = news["cluster_id"].astype(str)
    news["published_at"] = pd.to_datetime(news["published_at"], errors="coerce")

    if news["published_at"].isna().any():
        bad_rows = int(news["published_at"].isna().sum())
        raise ValueError(f"Found {bad_rows} rows with invalid published_at")

    normalized_embeddings = l2_normalize_np(embeddings)

    news["_row_pos"] = np.arange(len(news))
    exclude_context_ids = exclude_context_ids or set()

    rows = []

    sorted_news = news.sort_values(["cluster_id", "published_at", "_row_pos"], kind="mergesort")

    for cluster_id, cluster_df in sorted_news.groupby("cluster_id", sort=False):
        cluster_df = (
            cluster_df
            .sort_values(["published_at", "_row_pos"], kind="mergesort")
            .reset_index(drop=True)
        )

        if len(cluster_df) <= 1:
            continue

        for position_zero_based, current_row in cluster_df.iterrows():
            current_news_id = str(current_row["news_id"])

            if current_news_id in exclude_current_ids:
                continue

            # First item has no within-cluster previous context.
            if position_zero_based == 0:
                continue

            previous_df = cluster_df.iloc[:position_zero_based].copy()

            if exclude_context_ids:
                previous_df = previous_df[
                    ~previous_df["news_id"].astype(str).isin(exclude_context_ids)
                ].copy()

            if previous_df.empty:
                continue

            current_embedding = normalized_embeddings[int(current_row["_row_pos"])]

            previous_indices = previous_df["_row_pos"].astype(int).to_numpy()
            previous_embeddings = normalized_embeddings[previous_indices]

            similarities = previous_embeddings @ current_embedding

            previous_df = previous_df.copy()
            previous_df["_similarity"] = similarities
            previous_df["_days_before"] = (
                current_row["published_at"] - previous_df["published_at"]
            ).dt.total_seconds() / 86400.0

            # Most similar previous items.
            previous_top = previous_df.sort_values(
                ["_similarity", "published_at"],
                ascending=[False, False],
            ).head(max_previous_items)

            # Ensure immediate previous item is present if possible.
            immediate_previous = previous_df.sort_values(
                ["published_at", "_row_pos"],
                ascending=[False, False],
            ).head(1)

            previous_top = (
                pd.concat([previous_top, immediate_previous], ignore_index=True)
                .drop_duplicates(subset=["news_id"], keep="first")
                .sort_values(["_similarity", "published_at"], ascending=[False, False])
                .head(max_previous_items)
            )

            previous_items = []

            for _, previous_row in previous_top.iterrows():
                previous_items.append({
                    "news_id": str(previous_row["news_id"]),
                    "published_at": str(previous_row["published_at"]),
                    "title": truncate_text(previous_row.get("title", ""), 220),
                    "text_snippet": truncate_text(get_text_value(previous_row), max_text_chars),
                    "similarity": round(float(previous_row["_similarity"]), 4),
                    "days_before": round(float(previous_row["_days_before"]), 2),
                })

            if not previous_items:
                continue

            max_prev_similarity = float(previous_df["_similarity"].max())
            last_previous_date = previous_df["published_at"].max()
            days_since_previous = float((current_row["published_at"] - last_previous_date).total_seconds() / 86400.0)

            rows.append({
                "news_id": current_news_id,
                "published_at": str(current_row["published_at"]),
                "topic": current_row.get("topic", ""),
                "cluster_id": str(cluster_id),
                "position_in_cluster": int(position_zero_based + 1),
                "cluster_size": int(len(cluster_df)),
                "title": truncate_text(current_row.get("title", ""), 220),
                "text_snippet": truncate_text(get_text_value(current_row), max_text_chars),
                "previous_items": json.dumps(previous_items, ensure_ascii=False),
                "max_prev_similarity": round(max_prev_similarity, 4),
                "days_since_previous": round(days_since_previous, 2),
                "bronze_label": "",
                "confidence": "",
                "needs_review": "",
                "bronze_comment": "",
            })

    candidates = pd.DataFrame(rows)

    if candidates.empty:
        return candidates

    # Balanced-ish deterministic selection by topic and similarity bucket.
    candidates["similarity_bucket"] = pd.cut(
        candidates["max_prev_similarity"],
        bins=[-np.inf, 0.75, 0.82, 0.88, 0.90, 0.94, np.inf],
        labels=["very_low", "low", "medium", "borderline", "high", "very_high"],
    ).astype(str)

    candidates = candidates.sample(
        frac=1.0,
        random_state=random_state,
    ).sort_values(
        ["topic", "similarity_bucket", "published_at", "cluster_id", "position_in_cluster"],
        kind="mergesort",
    )

    if len(candidates) > max_rows:
        selected_parts = []
        group_columns = ["topic", "similarity_bucket"]
        groups = list(candidates.groupby(group_columns, sort=False))
        per_group_target = max(1, int(np.ceil(max_rows / max(len(groups), 1))))

        for _, group_df in groups:
            selected_parts.append(group_df.head(per_group_target))

        selected = pd.concat(selected_parts, ignore_index=True)

        if len(selected) < max_rows:
            remaining = candidates[~candidates["news_id"].isin(selected["news_id"])].head(max_rows - len(selected))
            selected = pd.concat([selected, remaining], ignore_index=True)

        candidates = selected.head(max_rows).copy()

    candidates = candidates.drop(columns=["similarity_bucket"])
    candidates = candidates.sort_values(
        ["topic", "cluster_id", "published_at", "position_in_cluster"],
        kind="mergesort",
    ).reset_index(drop=True)

    return candidates


def save_bronze_batches(
    candidates: pd.DataFrame,
    output_dir: Path,
    batch_size: int = 100,
) -> list[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)

    for old_file in output_dir.glob("bronze_candidates_batch_*.csv"):
        old_file.unlink()

    batch_paths = []

    for batch_idx, start in enumerate(range(0, len(candidates), batch_size)):
        batch = candidates.iloc[start:start + batch_size].copy()

        batch_path = output_dir / f"bronze_candidates_batch_{batch_idx:03d}.csv"
        batch.to_csv(batch_path, index=False, encoding="utf-8-sig")

        batch_paths.append(batch_path)

    return batch_paths


def write_bronze_instructions(path: Path) -> None:
    instructions = """# Bronze annotation instructions

Task: label whether the current news article adds meaningful novelty compared with previous articles from the same story cluster.

Use only the provided current article and previous_items context.

Labels:

- significant: the article adds a new meaningful fact, event, decision, participant, consequence, number, date, or substantial development.
- minor: the article gives a small clarification or weak update, but mostly continues already known information.
- duplicate: the article is essentially a repeat, rewrite, repost, or near-duplicate of previous information.
- wrong_cluster: the article does not belong to the same story as the previous context.
- unclear: there is not enough context to decide confidently.

Confidence:

- high: label is clear from the text.
- medium: label is plausible but there is some ambiguity.
- low: weak confidence; should usually be excluded from training.

needs_review:

- True if the row should be manually checked.
- False if the label is usable as bronze training data.

Expected output columns:

news_id, cluster_id, bronze_label, confidence, needs_review, bronze_comment

Important:
- Do not use golden/silver labels.
- Do not infer facts not present in the current row or previous_items.
- Prefer duplicate when the current article mainly repeats the same facts.
- Prefer significant when a new concrete fact changes the story state.
"""
    path.write_text(instructions, encoding="utf-8")


def write_manifest(
    path: Path,
    candidates: pd.DataFrame,
    years: list[int],
    max_news_per_year,
    golden_ids: set[str],
    silver_ids: set[str],
    current_candidate_ids: set[str],
    batch_paths: list[Path],
) -> None:
    manifest = {
        "candidate_rows": int(len(candidates)),
        "unique_candidate_ids": int(candidates["news_id"].nunique()),
        "batch_count": int(len(batch_paths)),
        "batch_size": int(LARGE_BRONZE_BATCH_SIZE),
        "source_years": years,
        "max_news_per_year": max_news_per_year,
        "golden_ids_excluded": int(len(golden_ids)),
        "silver_ids_excluded": int(len(silver_ids)),
        "current_candidate_ids_excluded": int(len(current_candidate_ids)),
        "story_threshold": STORY_THRESHOLD,
        "story_window_days": STORY_WINDOW_DAYS,
        "strict_context_excludes": bool(LARGE_BRONZE_STRICT_CONTEXT_EXCLUDES),
        "columns": list(candidates.columns),
        "label_columns_to_fill": [
            "bronze_label",
            "confidence",
            "needs_review",
            "bronze_comment",
        ],
    }

    path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")


def zip_package(
    package_path: Path,
    files: list[Path],
    root_dir: Path,
) -> None:
    if package_path.exists():
        package_path.unlink()

    with zipfile.ZipFile(package_path, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
        for file_path in files:
            archive.write(file_path, arcname=file_path.relative_to(root_dir))

print("Helpers loaded.")

## 4. Сбор no-leakage ids

Исключаем из bronze current rows:

- human `golden`;
- LLM/silver train rows;
- весь текущий `lenta_golden_candidate_pool.csv`, чтобы bronze был из вообще незадействованной части Lenta.

In [ ]:
# ---------------------------------------------------------------------
# Collect no-leakage ids
# ---------------------------------------------------------------------

golden_path = paths.golden_path
current_candidate_pool_path = paths.prepared_dir / "lenta_golden_candidate_pool.csv"

if not golden_path.exists():
    raise FileNotFoundError(f"Golden file not found: {golden_path}")

if not current_candidate_pool_path.exists():
    raise FileNotFoundError(f"Candidate pool file not found: {current_candidate_pool_path}")

golden = pd.read_csv(golden_path)
current_candidate_pool = pd.read_csv(current_candidate_pool_path)

golden_ids = collect_ids_from_frame(golden, "golden")
silver_ids = collect_silver_ids(paths)
current_candidate_ids = collect_ids_from_frame(current_candidate_pool, "current_candidate_pool")

large_bronze_exclude_ids = golden_ids | silver_ids | current_candidate_ids

print("
Excluded ids:")
print("  golden:", len(golden_ids))
print("  silver:", len(silver_ids))
print("  current candidate pool:", len(current_candidate_ids))
print("  total:", len(large_bronze_exclude_ids))

## 5. Загрузка данных из других годов Lenta

In [ ]:
# ---------------------------------------------------------------------
# Load large raw Lenta slice from unused years
# ---------------------------------------------------------------------

if not LARGE_BRONZE_INPUT_PATH.exists():
    raise FileNotFoundError(f"Lenta clean file not found: {LARGE_BRONZE_INPUT_PATH}")

large_bronze_raw_all = pd.read_csv(LARGE_BRONZE_INPUT_PATH)

large_bronze_raw_all["news_id"] = normalize_news_id_for_join(large_bronze_raw_all["news_id"])
large_bronze_raw_all["published_at"] = pd.to_datetime(
    large_bronze_raw_all["published_at"],
    errors="coerce",
)

large_bronze_raw_all = large_bronze_raw_all[
    large_bronze_raw_all["published_at"].notna()
].copy()

large_bronze_raw_all["year"] = large_bronze_raw_all["published_at"].dt.year

large_bronze_raw = large_bronze_raw_all[
    large_bronze_raw_all["year"].isin(LARGE_BRONZE_YEARS)
].copy()

before_exclude = len(large_bronze_raw)

large_bronze_raw = large_bronze_raw[
    ~large_bronze_raw["news_id"].isin(large_bronze_exclude_ids)
].copy()

after_exclude = len(large_bronze_raw)

print("Rows before exclude:", before_exclude)
print("Rows after exclude:", after_exclude)
print("Years:")
display(large_bronze_raw["year"].value_counts().sort_index())
print("Topics:")
display(large_bronze_raw["topic"].value_counts().head(20))

if large_bronze_raw.empty:
    raise RuntimeError("No rows left for large bronze after exclusions.")

## 6. Embeddings + legacy clustering по годам

Годы обрабатываются по отдельности, чтобы не запускать слишком тяжёлую кластеризацию на всём датасете сразу. Embeddings кэшируются по году.

In [ ]:
# ---------------------------------------------------------------------
# Encode and cluster year by year
# ---------------------------------------------------------------------

large_bronze_clustered_parts = []
large_bronze_embedding_parts = []
large_bronze_year_summaries = []

for year in LARGE_BRONZE_YEARS:
    year_raw = large_bronze_raw[large_bronze_raw["year"] == year].copy()

    if year_raw.empty:
        print(f"
Year {year}: no rows, skipped")
        continue

    if MAX_NEWS_PER_YEAR is not None and len(year_raw) > MAX_NEWS_PER_YEAR:
        year_raw = (
            year_raw
            .sample(n=MAX_NEWS_PER_YEAR, random_state=LARGE_BRONZE_RANDOM_STATE)
            .sort_values(["published_at", "news_id"], kind="mergesort")
            .reset_index(drop=True)
        )

    year_input = prepare_clean_like_for_bronze(year_raw)

    print(f"
Year {year}: input rows = {len(year_input)}")

    year_cache_path = (
        paths.artifacts_dir
        / "embeddings"
        / f"bge_m3_large_bronze_{year}_id_aligned.npz"
    )
    year_cache_path.parent.mkdir(parents=True, exist_ok=True)

    year_embeddings = encoder.encode_dataframe(
        year_input,
        text_column="model_text",
        id_column="news_id",
        cache_path=year_cache_path,
        force_recompute=False,
    )

    year_cluster_ids, edge_count = legacy_threshold_graph_cluster_ids(
        news_df=year_input,
        embeddings=year_embeddings,
        similarity_threshold=STORY_THRESHOLD,
        window_days=STORY_WINDOW_DAYS,
        use_topic_blocking=USE_TOPIC_BLOCKING,
    )

    year_clustered = year_input.copy()
    year_clustered["cluster_id"] = (
        "bronze_"
        + str(year)
        + "_"
        + year_cluster_ids.astype(str)
    )

    cluster_sizes = year_clustered.groupby("cluster_id").size()
    non_singleton_rows = int(cluster_sizes[cluster_sizes > 1].sum())
    max_non_first_candidates = int((cluster_sizes - 1).clip(lower=0).sum())

    print("  embeddings:", year_embeddings.shape)
    print("  graph edges:", edge_count)
    print("  clusters:", year_clustered["cluster_id"].nunique())
    print("  non-singleton rows:", non_singleton_rows)
    print("  max non-first candidates:", max_non_first_candidates)

    large_bronze_clustered_parts.append(year_clustered)
    large_bronze_embedding_parts.append(year_embeddings)
    large_bronze_year_summaries.append({
        "year": int(year),
        "rows": int(len(year_clustered)),
        "edges": int(edge_count),
        "clusters": int(year_clustered["cluster_id"].nunique()),
        "non_singleton_rows": non_singleton_rows,
        "max_non_first_candidates": max_non_first_candidates,
        "cache_path": str(year_cache_path),
    })

if not large_bronze_clustered_parts:
    raise RuntimeError("No clustered bronze parts were produced.")

large_bronze_clustered_news = pd.concat(large_bronze_clustered_parts, ignore_index=True)
large_bronze_embeddings = np.vstack(large_bronze_embedding_parts)

print("
Large bronze clustered news:", large_bronze_clustered_news.shape)
print("Large bronze embeddings:", large_bronze_embeddings.shape)
print("Large bronze clusters:", large_bronze_clustered_news["cluster_id"].nunique())

display(pd.DataFrame(large_bronze_year_summaries))

## 7. Сбор bronze candidates и экспорт пакета

In [ ]:
# ---------------------------------------------------------------------
# Build annotation candidates and export package
# ---------------------------------------------------------------------

context_exclude_ids = (
    large_bronze_exclude_ids
    if LARGE_BRONZE_STRICT_CONTEXT_EXCLUDES
    else set()
)

large_bronze_candidates = build_bronze_annotation_candidates(
    clustered_news=large_bronze_clustered_news,
    embeddings=large_bronze_embeddings,
    exclude_current_ids=large_bronze_exclude_ids,
    exclude_context_ids=context_exclude_ids,
    max_rows=LARGE_BRONZE_MAX_CANDIDATES,
    max_previous_items=LARGE_BRONZE_MAX_PREVIOUS_ITEMS,
    max_text_chars=LARGE_BRONZE_MAX_TEXT_CHARS,
    random_state=LARGE_BRONZE_RANDOM_STATE,
)

if large_bronze_candidates.empty:
    raise RuntimeError("No large bronze candidates were produced.")

# Hard no-leakage check for current rows.
candidate_ids = set(normalize_news_id_for_join(large_bronze_candidates["news_id"]))

current_golden_leaks = candidate_ids & golden_ids
current_silver_leaks = candidate_ids & silver_ids
current_old_candidate_leaks = candidate_ids & current_candidate_ids

if current_golden_leaks:
    raise AssertionError(f"Golden leakage in current rows: {len(current_golden_leaks)}")

if current_silver_leaks:
    raise AssertionError(f"Silver leakage in current rows: {len(current_silver_leaks)}")

if current_old_candidate_leaks:
    raise AssertionError(f"Old candidate pool leakage in current rows: {len(current_old_candidate_leaks)}")

# Optional strict context leakage check.
if LARGE_BRONZE_STRICT_CONTEXT_EXCLUDES:
    context_ids = set()

    for value in large_bronze_candidates["previous_items"]:
        for item in json.loads(value):
            context_ids.add(str(item["news_id"]))

    context_leaks = context_ids & large_bronze_exclude_ids

    if context_leaks:
        raise AssertionError(f"Excluded ids leakage in previous_items: {len(context_leaks)}")

LARGE_BRONZE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LARGE_BRONZE_BATCH_DIR.mkdir(parents=True, exist_ok=True)

large_all_candidates_path = LARGE_BRONZE_OUTPUT_DIR / "bronze_large_candidates_all.csv"
large_instructions_path = LARGE_BRONZE_OUTPUT_DIR / "bronze_large_annotation_instructions.md"
large_manifest_path = LARGE_BRONZE_OUTPUT_DIR / "bronze_large_manifest.json"

large_bronze_candidates.to_csv(
    large_all_candidates_path,
    index=False,
    encoding="utf-8-sig",
)

large_batch_paths = save_bronze_batches(
    candidates=large_bronze_candidates,
    output_dir=LARGE_BRONZE_BATCH_DIR,
    batch_size=LARGE_BRONZE_BATCH_SIZE,
)

write_bronze_instructions(large_instructions_path)
write_manifest(
    path=large_manifest_path,
    candidates=large_bronze_candidates,
    years=LARGE_BRONZE_YEARS,
    max_news_per_year=MAX_NEWS_PER_YEAR,
    golden_ids=golden_ids,
    silver_ids=silver_ids,
    current_candidate_ids=current_candidate_ids,
    batch_paths=large_batch_paths,
)

package_files = [
    large_all_candidates_path,
    large_instructions_path,
    large_manifest_path,
    *large_batch_paths,
]

zip_package(
    package_path=LARGE_BRONZE_PACKAGE_PATH,
    files=package_files,
    root_dir=LARGE_BRONZE_OUTPUT_DIR,
)

print("DONE")
print("Large bronze candidates:", large_bronze_candidates.shape)
print("Unique candidate ids:", large_bronze_candidates["news_id"].nunique())
print("Batches:", len(large_batch_paths))
print("All candidates:", large_all_candidates_path)
print("Instructions:", large_instructions_path)
print("Manifest:", large_manifest_path)
print("Zip package:", LARGE_BRONZE_PACKAGE_PATH)

print("
Leakage check for current rows:")
print("  ∩ golden:", len(current_golden_leaks))
print("  ∩ silver:", len(current_silver_leaks))
print("  ∩ old candidate pool:", len(current_old_candidate_leaks))

display(large_bronze_candidates.head(10))

## 8. Быстрый просмотр первого batch

После проверки первых 1–2 batch-файлов их можно отправлять на разметку. Если формат хороший, увеличиваем `LARGE_BRONZE_MAX_CANDIDATES` / годы.

In [ ]:
# ---------------------------------------------------------------------
# Preview first batch
# ---------------------------------------------------------------------

if "large_batch_paths" in globals() and large_batch_paths:
    first_batch = pd.read_csv(large_batch_paths[0])
    print("First batch:", large_batch_paths[0])
    print("Rows:", first_batch.shape)
    display(first_batch.head(5))
else:
    print("No batch paths found. Run the export cell first.")